# SoccerNet MVFouls — Direct-Clip Multi-Task Referee v2

One incident-level video model reads original MP4 clips and predicts offence,
card/no-card severity, a supportable action family, and body-part context.
Version 2 responds directly to the first run: higher 160×160 resolution,
24-frame clips, learned camera-view attention, consolidated rare labels, and
an imbalance strategy that does not double-weight the offence decision.
The official test is locked during development.

In [ ]:
# Run once if the environment is missing packages, then restart the kernel.
# %pip install -q torch torchvision opencv-python-headless pandas scikit-learn matplotlib

In [ ]:
import copy, json, random, time
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score, recall_score)
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision.models.video import R3D_18_Weights, r3d_18

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else
                      "mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
@dataclass
class Config:
    frames: int = 24
    center_coverage: float = 0.70
    views_per_incident: int = 3
    image_size: int = 160
    batch_incidents: int = 2
    workers: int = 2
    epochs: int = 20
    patience: int = 5
    backbone_lr: float = 1e-5
    heads_lr: float = 3e-4
    weight_decay: float = 1e-4
    dropout: float = 0.35
    grad_clip: float = 5.0

cfg = Config()
RUN_TRAINING = False
RUN_FINAL_TEST = False

davis = Path("/home/cosmos32/AIreferee/data/mvfouls")
mac = Path("/Users/rishideshpande/Downloads/data/soccernet/mvfouls")
dataset_path = davis if davis.exists() else mac
if not dataset_path.exists(): raise FileNotFoundError("MVFouls was not found.")
artifact_dir = Path("artifacts/multitask_video")
checkpoint_path = artifact_dir / "best_multitask_r3d18.pt"
history_path = artifact_dir / "history.csv"
print("Dataset:", dataset_path)
print("Safety switches:", RUN_TRAINING, RUN_FINAL_TEST)

## Labels that the dataset can support

The first run proved that five-way severity/action detail was too sparse.
This version uses `No card` versus `Card` and four action families. Red cards,
dives, and handballs remain represented, but are not asked to form reliable
standalone classes from only a few dozen examples. Unknown fields are masked.

In [ ]:
SEVERITY_NAMES = ["No card", "Card"]
ACTION_NAMES = ["Tackle", "Physical challenge", "Dangerous play", "Other"]
ACTION_GROUP = {
    "Standing tackling":0, "Tackling":0,
    "Challenge":1, "Holding":1, "Pushing":1,
    "Elbowing":2, "High leg":2,
    "Dive":3, "Handball":3,
}
BODY_NAMES = ["Under body", "Upper body"]

def severity_id(value):
    if value in {"1.0","2.0"}: return 0
    if value in {"3.0","4.0","5.0"}: return 1
    return -100

def read_split(root, split):
    actions=json.loads((root/split/"annotations.json").read_text())["Actions"]
    rows=[]
    for action_id,a in actions.items():
        if a.get("Offence") not in {"Offence","No offence"}: continue
        paths=sorted((root/split/f"action_{action_id}").glob("*.mp4"))
        if not paths: continue
        action="Handball" if a.get("Handball")=="Handball" else a.get("Action class","")
        offence=int(a["Offence"]=="Offence")
        rows.append(dict(split=split,action_id=str(action_id),
            video_paths=[str(x) for x in paths],offence=offence,
            severity=severity_id(a.get("Severity","")) if offence else -100,
            action=ACTION_GROUP.get(action,-100),
            body={"Under body":0,"Upper body":1}.get(a.get("Bodypart"),-100)))
    return rows

splits={s:read_split(dataset_path,s) for s in ("train","valid","test")}
for name,rows in splits.items():
    print("\n",name,"incidents",len(rows),"clips",sum(len(r["video_paths"]) for r in rows))
    for field in ("offence","severity","action","body"):
        print(field,Counter(r[field] for r in rows if r[field] != -100))

In [ ]:
def sample_indices(total, count, coverage, training=False):
    width = max(count, min(total, round(total*coverage)))
    center = (total-1)/2
    if training:
        center += random.uniform(-0.10,0.10)*total
    lo = max(0, min(total-width, round(center-width/2)))
    return np.linspace(lo, lo+width-1, count).round().astype(int)

def decode_clip(path, training=False):
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0: cap.release(); raise ValueError(f"Unreadable clip: {path}")
    frames = []
    for idx in sample_indices(total,cfg.frames,cfg.center_coverage,training):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(idx)); ok,bgr = cap.read()
        if not ok: cap.release(); raise ValueError(f"Decode failed: {path}")
        frames.append(cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    cap.release()
    x = torch.from_numpy(np.stack(frames)).permute(0,3,1,2)
    # Official Kinetics preprocessing returns [C,T,H,W].
    x=x.float().div(255.0)
    x=torch.nn.functional.interpolate(x,size=(cfg.image_size,cfg.image_size),
                                      mode="bilinear",align_corners=False)
    mean=torch.tensor([0.43216,0.394666,0.37645]).view(1,3,1,1)
    std=torch.tensor([0.22803,0.22145,0.216989]).view(1,3,1,1)
    x=(x-mean)/std
    return x.permute(1,0,2,3).contiguous()

class IncidentDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows, self.training = rows, training
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]; paths = list(r["video_paths"])
        if self.training: random.shuffle(paths)
        paths = paths[:cfg.views_per_incident]
        views = torch.stack([decode_clip(p,self.training) for p in paths])
        labels = {k:torch.tensor(r[k]) for k in ("offence","severity","action","body")}
        return views, labels, r["action_id"], paths

def collate_incidents(batch):
    views=[]; owners=[]; labels={k:[] for k in ("offence","severity","action","body")}
    ids=[]; paths=[]
    for owner,(v,y,aid,p) in enumerate(batch):
        views.append(v); owners.extend([owner]*len(v)); ids.append(aid); paths.append(p)
        for k in labels: labels[k].append(y[k])
    return (torch.cat(views),torch.tensor(owners),
            {k:torch.stack(v) for k,v in labels.items()},ids,paths)

# Decode one real local clip now; this is a safe data-pipeline check.
probe = IncidentDataset(splits["train"][:1])[0]
print("Probe views:",probe[0].shape, "labels:",{k:int(v) for k,v in probe[1].items()})

In [ ]:
class MultiTaskVideoReferee(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        weights = R3D_18_Weights.DEFAULT if pretrained else None
        net = r3d_18(weights=weights)
        net.fc = nn.Identity()
        self.backbone = net
        # Freeze generic early motion features; fine-tune layer3/layer4.
        for module in (net.stem,net.layer1,net.layer2):
            for p in module.parameters(): p.requires_grad=False
        self.norm = nn.LayerNorm(512)
        self.view_attention = nn.Sequential(nn.Linear(512,128),nn.Tanh(),nn.Linear(128,1))
        self.view_attention = nn.Sequential(nn.Linear(512,128),nn.Tanh(),nn.Linear(128,1))
        self.view_attention = nn.Sequential(nn.Linear(512,128),nn.Tanh(),nn.Linear(128,1))
        def head(n): return nn.Sequential(nn.Dropout(cfg.dropout),nn.Linear(512,256),
                                          nn.GELU(),nn.Dropout(cfg.dropout),nn.Linear(256,n))
        self.heads = nn.ModuleDict(dict(offence=head(2),severity=head(2),
            action=head(len(ACTION_NAMES)),body=head(2)))
    def train(self, mode=True):
        super().train(mode)
        if mode:
            self.backbone.stem.eval(); self.backbone.layer1.eval(); self.backbone.layer2.eval()
        return self
    def forward(self, views, owners, incidents):
        z = self.backbone(views)
        pooled=[]
        scores=self.view_attention(z).squeeze(1)
        for incident in range(incidents):
            mask=owners==incident
            weights=torch.softmax(scores[mask],dim=0)
            pooled.append((z[mask]*weights.unsqueeze(1)).sum(0))
        pooled=self.norm(torch.stack(pooled))
        return {k:h(pooled) for k,h in self.heads.items()}

# Shape check without downloading weights again.
shape_model = MultiTaskVideoReferee(pretrained=False).eval()
with torch.inference_mode():
    fake = torch.zeros(2,3,cfg.frames,cfg.image_size,cfg.image_size)
    out = shape_model(fake,torch.tensor([0,0]),1)
print({k:tuple(v.shape) for k,v in out.items()})
del shape_model

In [ ]:
def capped_weights(rows, field, classes, cap=4.0):
    y=[r[field] for r in rows if r[field] != -100]
    counts=np.bincount(y,minlength=classes)
    w=len(y)/(classes*np.maximum(counts,1))
    w=w/w.min()
    return torch.tensor(np.minimum(w,cap),dtype=torch.float32,device=device)

LOSS_SCALE=dict(offence=1.0,severity=.45,action=.40,body=.25)

def make_losses(rows):
    # The sampler already balances offence, so weighting it again would
    # over-correct. Auxiliary weights are capped to prevent tiny classes
    # from dominating gradients.
    return {
      "offence":nn.CrossEntropyLoss(label_smoothing=.05),
      "severity":nn.CrossEntropyLoss(weight=capped_weights(rows,"severity",2),label_smoothing=.03),
      "action":nn.CrossEntropyLoss(weight=capped_weights(rows,"action",len(ACTION_NAMES)),label_smoothing=.03),
      "body":nn.CrossEntropyLoss(weight=capped_weights(rows,"body",2),label_smoothing=.03)}

def loaders():
    labels=np.array([r["offence"] for r in splits["train"]])
    counts=np.bincount(labels,minlength=2)
    sample_weights=np.array([1/counts[y] for y in labels])
    sampler=WeightedRandomSampler(sample_weights,len(sample_weights),replacement=True)
    common=dict(batch_size=cfg.batch_incidents,num_workers=cfg.workers,
                collate_fn=collate_incidents,pin_memory=device.type=="cuda")
    return {
      "train":DataLoader(IncidentDataset(splits["train"],True),sampler=sampler,**common),
      "valid":DataLoader(IncidentDataset(splits["valid"]),shuffle=False,**common),
      "test":DataLoader(IncidentDataset(splits["test"]),shuffle=False,**common)}

def metric_block(y,p):
    return dict(accuracy=accuracy_score(y,p),
        balanced_accuracy=balanced_accuracy_score(y,p),
        macro_f1=f1_score(y,p,average="macro",zero_division=0))

def run_epoch(model,loader,losses,optimizer=None):
    training=optimizer is not None; model.train(training)
    saved={k:{"y":[],"p":[],"logits":[]} for k in losses}; total=0; n=0
    for views,owners,labels,_,_ in loader:
        views,owners=views.to(device),owners.to(device)
        labels={k:v.to(device) for k,v in labels.items()}
        with torch.set_grad_enabled(training):
            logits=model(views,owners,len(labels["offence"]))
            terms=[]
            for k in losses:
                mask=labels[k] != -100
                if mask.any(): terms.append(LOSS_SCALE[k]*losses[k](logits[k][mask],labels[k][mask]))
            loss=sum(terms)
            if training:
                optimizer.zero_grad(set_to_none=True); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(),cfg.grad_clip); optimizer.step()
        total+=float(loss)*len(labels["offence"]); n+=len(labels["offence"])
        for k in losses:
            mask=labels[k] != -100
            saved[k]["y"].extend(labels[k][mask].cpu().numpy())
            saved[k]["p"].extend(logits[k][mask].argmax(1).cpu().numpy())
            saved[k]["logits"].append(logits[k][mask].detach().cpu())
    result={"loss":total/n,"raw":saved}
    for k,d in saved.items(): result[k]=metric_block(np.asarray(d["y"]),np.asarray(d["p"]))
    return result

In [ ]:
if RUN_TRAINING:
    if device.type != "cuda": raise RuntimeError("Train this video model on a Davis GPU.")
    artifact_dir.mkdir(parents=True,exist_ok=True)
    dl=loaders(); model=MultiTaskVideoReferee(pretrained=True).to(device)
    losses=make_losses(splits["train"])
    backbone=[p for n,p in model.named_parameters() if n.startswith("backbone") and p.requires_grad]
    heads=[p for n,p in model.named_parameters() if not n.startswith("backbone") and p.requires_grad]
    opt=torch.optim.AdamW([{"params":backbone,"lr":cfg.backbone_lr},
                           {"params":heads,"lr":cfg.heads_lr}],
                          weight_decay=cfg.weight_decay)
    best=-1; stale=0; history=[]
    for epoch in range(1,cfg.epochs+1):
        tr=run_epoch(model,dl["train"],losses,opt)
        va=run_epoch(model,dl["valid"],losses)
        row={"epoch":epoch,"train_loss":tr["loss"],"valid_loss":va["loss"]}
        for head in losses:
            for key,val in va[head].items(): row[f"valid_{head}_{key}"]=val
        history.append(row)
        score=(va["offence"]["balanced_accuracy"]+va["offence"]["macro_f1"])/2
        print(f'Epoch {epoch:02d} | loss {tr["loss"]:.4f} | '
              f'offence bal {score:.4f} F1 {va["offence"]["macro_f1"]:.4f} | '
              f'severity bal {va["severity"]["balanced_accuracy"]:.4f} | '
              f'action bal {va["action"]["balanced_accuracy"]:.4f} | score {score:.4f}')
        if score>best:
            best=score; stale=0
            torch.save({"model":copy.deepcopy(model.state_dict()),"config":asdict(cfg),
                        "epoch":epoch,"validation":row},checkpoint_path)
        else:
            stale+=1
            if stale>=cfg.patience: print("Early stopping"); break
    pd.DataFrame(history).to_csv(history_path,index=False)
    print("Best validation selection score:",best)
else:
    print("Training is OFF.")

In [ ]:
HEAD_NAMES=dict(offence=["No offence","Offence"],severity=SEVERITY_NAMES,
                action=ACTION_NAMES,body=BODY_NAMES)

def print_report(head, y, p):
    print("\n",head.upper())
    print(metric_block(y,p)); print(confusion_matrix(y,p))
    print(classification_report(y,p,target_names=HEAD_NAMES[head],zero_division=0))

if RUN_FINAL_TEST:
    if RUN_TRAINING: raise RuntimeError("Turn training off before final testing.")
    if not checkpoint_path.exists(): raise FileNotFoundError("No trained checkpoint.")
    dl=loaders(); ck=torch.load(checkpoint_path,map_location=device)
    model=MultiTaskVideoReferee(pretrained=False).to(device)
    model.load_state_dict(ck["model"]); losses=make_losses(splits["train"])
    test=run_epoch(model,dl["test"],losses)
    for head,d in test["raw"].items():
        print_report(head,np.asarray(d["y"]),np.asarray(d["p"]))
else:
    print("Final test is locked.")

In [ ]:
def predict_clips(video_paths, checkpoint=checkpoint_path):
    ck=torch.load(checkpoint,map_location=device)
    model=MultiTaskVideoReferee(pretrained=False).to(device)
    model.load_state_dict(ck["model"]); model.eval()
    views=torch.stack([decode_clip(str(p)) for p in video_paths]).to(device)
    with torch.inference_mode():
        logits=model(views,torch.zeros(len(views),dtype=torch.long,device=device),1)
    answer={}
    for head,names in HEAD_NAMES.items():
        probs=logits[head].softmax(1)[0].cpu().numpy(); idx=int(probs.argmax())
        answer[head]={"prediction":names[idx],"confidence":float(probs[idx]),
                      "probabilities":dict(zip(names,probs.round(4).tolist()))}
    return answer

# After training:
# prediction = predict_clips(["/path/to/clip_0.mp4", "/path/to/clip_1.mp4"])
# prediction

## Correct interpretation

- This version must be selected using validation only; the official test was
  already consumed by the first model.
- The primary result is offence balanced accuracy and macro-F1.
- Severity is now `No card` versus `Card`; there are too few red cards for an
  honest standalone red-card model.
- Action families are intentionally broader because rare nine-way classes
  could not be learned.
- Softmax confidence is not calibrated probability. Calibrate only after the
  classifier demonstrates useful validation performance.